# Exp7.3.7 — Whole-count Linear bias transfer to LIF count space

Analysis-only notebook. It reads finalized artifacts and shows aggregate method-level comparisons across the three seeds; it does not train models or launch jobs.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('artifacts/experiment_7_3_7_wholecount_bias_transfer/wholecount_bias_transfer_v1')
if not ROOT.exists():
    ROOT = Path('notebooks') / ROOT

methods = pd.read_csv(ROOT / 'method_summary.csv')
gaps = pd.read_csv(ROOT / 'gap_summary.csv')
bias = pd.read_csv(ROOT / 'bias_vector_summary.csv')
history = pd.read_csv(ROOT / 'calibration_history_summary.csv')
source_checks = json.loads((ROOT / 'source_reproduction_checks.json').read_text())
manifest = json.loads((ROOT / 'manifest.json').read_text())

print('Experiment:', manifest['experiment_id'])
print('Source reproduction:', source_checks)


## Final method comparison

`L1` is the affine whole-count Linear reference. `L2` transfers exactly the same raw Linear weight matrix to the beta=0.5 output LIF with no LIF weight training. `L3/L4` only add continuous/integer output-count offsets.


In [ ]:
cols = [
    'method', 'train_ba_mean', 'val_ba_mean', 'test_ba_mean', 'test_ba_std',
    'weight_l2_mean', 'bias_l2_mean', 'bias_span_mean'
]
methods[cols]


In [ ]:
plot_df = methods.set_index('method').loc[manifest['methods']].reset_index()
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(plot_df['method'], 100 * plot_df['test_ba_mean'], yerr=100 * plot_df['test_ba_std'], capsize=4)
ax.set_ylabel('Test balanced accuracy (%)')
ax.set_title('Exp7.3.7 method-level comparison')
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()


## Where the performance moves

These paired contrasts decompose the benefit of the Linear affine bias, the same-W Linear-to-LIF realization loss, and how much is recovered by fixed output-count bias.


In [ ]:
gaps


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(gaps['contrast'], gaps['delta_pp_mean'], yerr=gaps['delta_pp_std'], capsize=4)
ax.axhline(0, linewidth=1)
ax.set_ylabel('Paired test BA difference (pp)')
ax.set_title('Exp7.3.7 paired gap decomposition')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()


## Learned class-specific count offsets

The continuous count-space bias is gauge-fixed to zero mean for training. For integer virtual spikes, each seed is shifted so the minimum class offset is zero before rounding.


In [ ]:
bias


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = bias['class_index']
ax.errorbar(x, bias['lif_count_bias_nonnegative_mean'], yerr=bias['lif_count_bias_nonnegative_std'], marker='o', capsize=3, label='continuous non-negative q')
ax.plot(x, bias['lif_count_bias_integer_mean'], marker='s', label='integer virtual bias spikes')
ax.set_xlabel('Class index')
ax.set_ylabel('Output-count offset (spikes)')
ax.set_title('Aggregate learned output-count bias')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## Count-bias calibration trajectory

Only the aggregate trajectory across seeds is shown.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(history['epoch'], 100 * history['train_ba_mean'], label='train BA')
ax.plot(history['epoch'], 100 * history['val_ba_mean'], label='val BA')
ax.set_xlabel('Calibration epoch')
ax.set_ylabel('Balanced accuracy (%)')
ax.set_title('Count-bias calibration')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()
